# 02 — Detector Tuning

Tests three strobe-robust detection approaches against Claude-annotated ground truth.

Run from `tests/object_detection/` after completing `01_explore.ipynb`:
```
cd tests/object_detection
jupyter notebook 02_detector.ipynb
```

In [ ]:
import sys
import pathlib
import json

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

HERE = pathlib.Path().resolve()
sys.path.insert(0, str(HERE.parent.parent / 'src'))

from aprilcam.vision.color_classifier import ColorClassifier, DEFAULT_COLOR_RANGES

frame_paths = sorted(HERE.glob('frame_*.jpg'))
frames_bgr = [cv.imread(str(p)) for p in frame_paths]
print(f'Loaded {len(frames_bgr)} frames')

# Load ground truth from 01_explore
gt_path = HERE / 'ground_truth.json'
if not gt_path.exists():
    raise FileNotFoundError('Run 01_explore.ipynb first to generate ground_truth.json')
all_annotations = json.loads(gt_path.read_text())

# Consensus: most-seen box locations
consensus_path = HERE / 'consensus.json'
consensus = json.loads(consensus_path.read_text())
print(f'Ground truth colors: {list(consensus.keys())}')

## Scoring helpers

Precision/recall against consensus ground truth. A detection is a match if its center is within `match_radius` pixels of a GT box center.

In [ ]:
MATCH_RADIUS = 50  # px

def score_detections(detected_objs, consensus, match_radius=MATCH_RADIUS):
    """Score a list of ObjectRecord against consensus ground truth.
    Returns precision, recall, tp, fp, fn.
    """
    gt_centers = {color: (info['cx'], info['cy']) for color, info in consensus.items()}
    matched_gt = set()
    tp = 0
    fp = 0
    for obj in detected_objs:
        cx, cy = obj.center_px
        hit = False
        for color, (gcx, gcy) in gt_centers.items():
            if color in matched_gt:
                continue
            if np.hypot(cx - gcx, cy - gcy) <= match_radius:
                matched_gt.add(color)
                tp += 1
                hit = True
                break
        if not hit:
            fp += 1
    fn = len(gt_centers) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return {'precision': precision, 'recall': recall, 'tp': tp, 'fp': fp, 'fn': fn}

def score_all_frames(classifier_fn, frames, consensus):
    """Apply classifier_fn(frame) -> [ObjectRecord] to all frames; aggregate scores."""
    all_scores = []
    total_fp = 0
    total_tp = 0
    total_fn = 0
    for frame in frames:
        objs = classifier_fn(frame)
        s = score_detections(objs, consensus)
        all_scores.append(s)
        total_tp += s['tp']
        total_fp += s['fp']
        total_fn += s['fn']
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    avg_fp = total_fp / len(frames)
    return {'precision': precision, 'recall': recall, 'avg_fp_per_frame': avg_fp,
            'tp': total_tp, 'fp': total_fp, 'fn': total_fn}

# Playfield polygon filter (inset 60px from frame edges)
def make_playfield_poly(frame):
    h, w = frame.shape[:2]
    INSET = 60
    return np.array([[INSET,INSET],[w-INSET,INSET],[w-INSET,h-INSET],[INSET,h-INSET]],
                    dtype=np.float32).reshape(-1,1,2)

def filter_objs(objs, frame):
    poly = make_playfield_poly(frame)
    out = []
    for obj in objs:
        cx, cy = obj.center_px
        if cv.pointPolygonTest(poly, (float(cx), float(cy)), False) < 0:
            continue
        x, y, bw, bh = obj.bbox
        aspect = max(bw, bh) / max(min(bw, bh), 1)
        if aspect > 2.0 or min(bw, bh) < 15 or max(bw, bh) > 200:
            continue
        out.append(obj)
    return out

print('Scoring helpers ready')

## Baseline: current ColorClassifier (no normalization)

In [ ]:
clf_baseline = ColorClassifier(min_area=600, max_area=30000)

def detect_baseline(frame):
    raw = clf_baseline.classify(frame)
    return filter_objs(raw, frame)

baseline_score = score_all_frames(detect_baseline, frames_bgr, consensus)
print(f"Baseline:  precision={baseline_score['precision']:.3f}  recall={baseline_score['recall']:.3f}  "
      f"avg_fp/frame={baseline_score['avg_fp_per_frame']:.1f}")

## Approach A: Per-frame V-channel normalization

Divide the V channel by `np.percentile(V, 90)` before thresholding. This should remove the strobe-induced brightness variation.

In [ ]:
def normalize_v(frame_bgr):
    hsv = cv.cvtColor(frame_bgr, cv.COLOR_BGR2HSV).astype(np.float32)
    v = hsv[:, :, 2]
    p90 = np.percentile(v, 90)
    if p90 > 1:
        hsv[:, :, 2] = np.clip(v / p90 * 200, 0, 255)
    return cv.cvtColor(hsv.astype(np.uint8), cv.COLOR_HSV2BGR)

clf_a = ColorClassifier(min_area=600, max_area=30000)

def detect_approach_a(frame):
    norm = normalize_v(frame)
    raw = clf_a.classify(norm)
    return filter_objs(raw, frame)

score_a = score_all_frames(detect_approach_a, frames_bgr, consensus)
print(f"Approach A (V-norm):  precision={score_a['precision']:.3f}  recall={score_a['recall']:.3f}  "
      f"avg_fp/frame={score_a['avg_fp_per_frame']:.1f}")

## Approach B: Temporal voting

Detect on all frames independently, then only keep detections that appear in ≥ K frames at roughly the same location. Strobe noise creates inconsistent detections; real boxes appear every frame.

In [ ]:
MIN_VOTE_FRAMES = 8  # require detection in at least 8/12 frames
CLUSTER_RADIUS = 40  # px — same-location threshold

clf_b = ColorClassifier(min_area=600, max_area=30000)

def detect_approach_b(frames):
    """Temporal voting over all frames; returns ObjectRecord list for last frame."""
    # Collect all detections with frame index
    all_dets = []
    last_frame = frames[-1]
    for fi, frame in enumerate(frames):
        raw = clf_b.classify(frame)
        for obj in filter_objs(raw, frame):
            all_dets.append((fi, obj))

    # Cluster by location
    used = [False] * len(all_dets)
    clusters = []
    for i, (fi, obj) in enumerate(all_dets):
        if used[i]:
            continue
        cx, cy = obj.center_px
        cluster = [(fi, obj)]
        used[i] = True
        for j, (fj, obj2) in enumerate(all_dets):
            if used[j]:
                continue
            cx2, cy2 = obj2.center_px
            if np.hypot(cx-cx2, cy-cy2) <= CLUSTER_RADIUS:
                cluster.append((fj, obj2))
                used[j] = True
        clusters.append(cluster)

    # Keep clusters seen in >= MIN_VOTE_FRAMES unique frames
    stable = []
    for cluster in clusters:
        unique_frames = len(set(fi for fi, _ in cluster))
        if unique_frames >= MIN_VOTE_FRAMES:
            # Return the detection from the last frame in this cluster
            last = sorted(cluster, key=lambda x: x[0])[-1]
            stable.append(last[1])
    return stable

voted = detect_approach_b(frames_bgr)
score_b_single = score_detections(voted, consensus)
print(f"Approach B (voting):  precision={score_b_single['precision']:.3f}  recall={score_b_single['recall']:.3f}  "
      f"tp={score_b_single['tp']}  fp={score_b_single['fp']}  fn={score_b_single['fn']}")

## Approach C: V-norm + tighter HSV ranges

Combine V normalization with tighter S minimum (derived from `01_explore` HSV summary). This should reduce wood-grain false positives while keeping all boxes.

In [ ]:
# Load the measured S medians from 01_explore to inform the threshold
# If hsv_summary.csv exists, use S_med - 2*S_std as min saturation floor
import csv

csv_path = HERE / 'hsv_summary.csv'
if csv_path.exists():
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    min_s_values = [max(20, float(r['S_med']) - 2*float(r['S_std'])) for r in rows]
    S_MIN_C = int(np.percentile(min_s_values, 10))  # conservative floor
    print(f'From hsv_summary: S_MIN_C = {S_MIN_C}')
else:
    S_MIN_C = 30
    print(f'No hsv_summary.csv; using S_MIN_C = {S_MIN_C}')

# Build tighter ranges with the measured S floor
TIGHT_RANGES = {}
for color, ranges in DEFAULT_COLOR_RANGES.items():
    TIGHT_RANGES[color] = [((lo[0], S_MIN_C, lo[2]), hi) for lo, hi in ranges]

clf_c = ColorClassifier(color_ranges=TIGHT_RANGES, min_area=600, max_area=30000)

def detect_approach_c(frame):
    norm = normalize_v(frame)
    raw = clf_c.classify(norm)
    return filter_objs(raw, frame)

score_c = score_all_frames(detect_approach_c, frames_bgr, consensus)
print(f"Approach C (V-norm+tight S):  precision={score_c['precision']:.3f}  recall={score_c['recall']:.3f}  "
      f"avg_fp/frame={score_c['avg_fp_per_frame']:.1f}")

## Comparison summary

In [ ]:
results = [
    ('Baseline (no norm)', baseline_score['precision'], baseline_score['recall'], baseline_score['avg_fp_per_frame']),
    ('Approach A (V-norm)', score_a['precision'], score_a['recall'], score_a['avg_fp_per_frame']),
    ('Approach B (temporal)', score_b_single['precision'], score_b_single['recall'], score_b_single['fp']),
    ('Approach C (V-norm+tight S)', score_c['precision'], score_c['recall'], score_c['avg_fp_per_frame']),
]

print(f"{'Approach':<30s}  {'Precision':>9s}  {'Recall':>6s}  {'FP/frame':>8s}")
print('-' * 62)
for name, prec, rec, fp in results:
    print(f'{name:<30s}  {prec:9.3f}  {rec:6.3f}  {fp:8.1f}')

# Pick winner: maximize recall first, then precision
best = max(results, key=lambda r: (r[2], r[1]))
print(f'\nWinner: {best[0]}')

## Visualize winner on all frames

In [ ]:
# Choose which detect function to use based on winner
winner_name = best[0]
if 'A' in winner_name:
    detect_winner = detect_approach_a
elif 'B' in winner_name:
    detect_winner = lambda f: detect_approach_b(frames_bgr)  # uses all frames
elif 'C' in winner_name:
    detect_winner = detect_approach_c
else:
    detect_winner = detect_baseline

COLOR_BGR = {
    'red': (0,0,220), 'orange': (0,128,255), 'yellow': (0,220,220),
    'green': (0,200,0), 'teal': (180,180,0), 'cyan': (220,220,0),
    'blue': (220,0,0), 'purple': (180,0,180),
}

cols = 4
rows = (len(frames_bgr) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3))

for idx, (ax, frame) in enumerate(zip(axes.flat, frames_bgr)):
    vis = frame.copy()
    objs = detect_winner(frame)
    # Draw GT centers
    for color, info in consensus.items():
        cv.circle(vis, (int(info['cx']), int(info['cy'])), 8, (0,255,0), 2)
    # Draw detections
    for obj in objs:
        x, y, bw, bh = obj.bbox
        bgr = COLOR_BGR.get(obj.color, (200,200,200))
        cv.rectangle(vis, (x,y), (x+bw,y+bh), bgr, 2)
        cv.putText(vis, obj.color, (x, y-4), cv.FONT_HERSHEY_SIMPLEX, 0.4, bgr, 1)
    ax.imshow(cv.cvtColor(vis, cv.COLOR_BGR2RGB))
    ax.set_title(f'{frame_paths[idx].name} ({len(objs)} det)', fontsize=7)
    ax.axis('off')

for ax in axes.flat[len(frames_bgr):]:
    ax.axis('off')

plt.suptitle(f'Winner: {winner_name}  (green circles = GT centers)', fontsize=10)
plt.tight_layout()
plt.savefig(str(HERE / 'winner_detections.jpg'), dpi=80)
plt.show()

## Output: copy-paste block for color_classifier.py

Run this cell after choosing an approach. Paste the output into `src/aprilcam/vision/color_classifier.py`.

In [ ]:
# Winning color ranges (adjust if Approach C won and tightened S)
if 'C' in winner_name:
    winning_ranges = TIGHT_RANGES
    preprocess_note = '# Pre-process: normalize_v(frame) before classify()'
elif 'A' in winner_name:
    winning_ranges = DEFAULT_COLOR_RANGES
    preprocess_note = '# Pre-process: normalize_v(frame) before classify()'
else:
    winning_ranges = DEFAULT_COLOR_RANGES
    preprocess_note = '# No pre-processing needed'

print('# ===== COPY INTO color_classifier.py =====')
print(preprocess_note)
print('DEFAULT_COLOR_RANGES = {')
for color, ranges in winning_ranges.items():
    ranges_str = ', '.join(f'({lo}, {hi})' for lo, hi in ranges)
    print(f'    "{color}": [{ranges_str}],')
print('}')
print()
if 'A' in winner_name or 'C' in winner_name:
    print('# normalize_v helper to add to ColorClassifier.classify():')
    print('def _normalize_v(frame_bgr):')
    print('    hsv = cv.cvtColor(frame_bgr, cv.COLOR_BGR2HSV).astype(np.float32)')
    print('    v = hsv[:, :, 2]')
    print('    p90 = np.percentile(v, 90)')
    print('    if p90 > 1:')
    print('        hsv[:, :, 2] = np.clip(v / p90 * 200, 0, 255)')
    print('    return cv.cvtColor(hsv.astype(np.uint8), cv.COLOR_HSV2BGR)')